In [1]:
import os
from datetime import datetime, date
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType, TimestampType
from clickhouse_driver import Client


In [2]:
# Configuration
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
CLICKHOUSE_HOST = "clickhouse"
CLICKHOUSE_PORT = 9000
CLICKHOUSE_USER = "default"
CLICKHOUSE_PASSWORD = ""
CLICKHOUSE_DB = "bronze"


In [3]:
# Spark Session
spark = (SparkSession.builder
    .appName("OracleToClickHouseIngestion")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate())


In [4]:
# Schema Definition
schema_oracle = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("amount", DecimalType(18, 2), True),
    StructField("transaction_date", DateType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("status", StringType(), True)
])


In [5]:
# Definição dos parâmetros de conexão com base no contexto (.env)
oracle_host = "10.255.150.11"
oracle_port = "1521"
oracle_service = "bi.grupotracker.com.br"
oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
print(f"Iniciando teste de conexão com: {jdbc_url}")

try:
    # Teste simples usando a tabela DUAL do Oracle
    # A query valida se conseguimos executar SQL no banco
    test_query = "(SELECT 'Conexão OK' as status, sysdate as data_hora FROM dual)"

    df_test_oracle = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("dbtable", test_query)
        .option("user", oracle_user)
        .option("password", oracle_password)
        .load()
    )

    print("Schema detectado:")
    df_test_oracle.printSchema()

    print("Resultado da consulta:")
    df_test_oracle.show(truncate=False)

except Exception as e:
    print(f"Erro ao conectar no Oracle: {str(e)}")


Iniciando teste de conexão com: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br
Erro ao conectar no Oracle: An error occurred while calling o38.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=DGhzzLLRQgqpbi/Hoqhegg==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvid

In [6]:
# Consulta para listar todos os schemas (owner) e tabelas acessíveis
query_metadata = "(SELECT owner, table_name FROM all_tables ORDER BY owner, table_name)"

try:
    print("Lendo estrutura de tabelas e schemas do Oracle...")
    df_structure = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("dbtable", query_metadata)
        .option("user", oracle_user)
        .option("password", oracle_password)
        .load()
    )

    print("Amostra de tabelas encontradas:")
    df_structure.show(10, truncate=False)

    print("Contagem de tabelas por Schema:")
    df_structure.groupBy("owner").count().orderBy(F.col("count").desc()).show()

except Exception as e:
    print(f"Erro ao ler metadados: {str(e)}")


Lendo estrutura de tabelas e schemas do Oracle...
Erro ao ler metadados: An error occurred while calling o49.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=kxg8VcuiRDOAijRsEoj9Pw==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcD

In [7]:
# Instalação do clickhouse-connect (se necessário)
try:
    import clickhouse_connect
except ImportError:
    !pip install clickhouse-connect
    import clickhouse_connect

from pyspark.sql.functions import current_timestamp

# 1. Lista todos os schemas e tabelas disponíveis no Oracle para gerar ingestion dinâmica
schemas_tables_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("dbtable", "(SELECT owner, table_name FROM all_tables ORDER BY owner, table_name)")
    .option("user", oracle_user)
    .option("password", oracle_password)
    .load()
    .select(
        F.col("OWNER").alias("owner"),
        F.col("TABLE_NAME").alias("table_name"),
    )
)

display(schemas_tables_df)

schemas_tables = schemas_tables_df.collect()

# 2. Define um modelo/padrão de mapeamento (colunas destino em ClickHouse)
clickhouse_columns = [
    ("id", "UInt64"),
    ("nome", "String"),
    ("valor", "Float64"),
    ("importado_em", "DateTime"),
]


def get_mapped_df(df):
    return (
        df
        .withColumn("importado_em", current_timestamp())
        .withColumnRenamed("transaction_id", "id")
        .withColumnRenamed("customer_id", "nome")
        .withColumnRenamed("amount", "valor")
        .select("id", "nome", "valor", "importado_em")
    )


def generate_ddl(database, table, columns):
    cols = ",\n    ".join([f"{col} {dtype}" for col, dtype in columns])
    return f"""
        CREATE TABLE IF NOT EXISTS {database}.{table}
        (
            {cols}
        ) ENGINE = MergeTree()
        ORDER BY id
    """


if __name__ == '__main__':
    client = clickhouse_connect.get_client(
        host='fxo48y5409.us-east1.gcp.clickhouse.cloud',
        user='default',
        password='1cb92kn~fcqO7',
        secure=True
    )
    client.command("CREATE DATABASE IF NOT EXISTS stage_demo")

    for row in schemas_tables:
        print(row.asDict())  # Adicione esta linha para inspecionar quais campos existem
        # continue com o uso correto do campo...
        schema = row['owner']
        table = row['table_name']
        oracle_full_table = f"{schema}.{table}"

        try:
            # UTILIZANDO O SPARK PARA LER DO ORACLE
            df_oracle = (
                spark.read.format("jdbc")
                .option("url", jdbc_url)
                .option("driver", "oracle.jdbc.OracleDriver")
                .option("dbtable", oracle_full_table)
                .option("user", oracle_user)
                .option("password", oracle_password)
                .load()
            )

            # Checa se as colunas necessárias existem antes de mapear
            required_cols = {'transaction_id', 'customer_id', 'amount'}
            if not required_cols.issubset(set(df_oracle.columns)):
                print(f"Pulando {oracle_full_table}: colunas obrigatórias não encontradas")
                continue

            # PROCESSAMENTO E MAPEAMENTO VIA SPARK
            df_clickhouse = get_mapped_df(df_oracle)
            stage_table = f"staging_{schema.lower()}_{table.lower()}"
            ddl = generate_ddl("stage_demo", stage_table, clickhouse_columns)
            client.command(ddl)
            print(f"Tabela {stage_table} criada/valida no ClickHouse.")

            # ENVIANDO OS DADOS VIA SPARK, CASO NECESSÁRIO USE SPARK-TURN CLICKHOUSE JDBC CONN
            # Aqui converte para pandas apenas para exemplo prático simples, mas normalmente deveria ser .write.format("jdbc")
            data_to_insert = df_clickhouse.toPandas().itertuples(index=False, name=None)
            insert_query = f"INSERT INTO stage_demo.{stage_table} (id, nome, valor, importado_em) VALUES"
            client.insert(insert_query, list(data_to_insert))
            print(f"Dados de {oracle_full_table} enviados para {stage_table}!")

            row_count = client.query(f"SELECT count(*) FROM stage_demo.{stage_table}").result_set[0][0]
            print(f"{stage_table}: {row_count} linhas no ClickHouse.")

        except Exception as e:
            print(f"Erro processando {oracle_full_table}: {str(e)}")


Py4JJavaError: An error occurred while calling o60.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=0URUQCJSRyWmSdjOPw8DZg==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.sca

In [ ]:
# Staging (Simulating Write to Shared Storage/Object Store)
staging_path = "/app/temp/staging_transactions"
df_oracle.write.mode("overwrite").parquet(staging_path)


In [ ]:
# ClickHouse Connection
client = Client(host=CLICKHOUSE_HOST, port=CLICKHOUSE_PORT, user=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD)


In [ ]:
# DDL Execution (Bronze Layer)
client.execute(f"CREATE DATABASE IF NOT EXISTS {CLICKHOUSE_DB}")

ddl_bronze = f"""
CREATE TABLE IF NOT EXISTS {CLICKHOUSE_DB}.transactions_local
(
    transaction_id String,
    customer_id UInt32,
    amount Decimal(18,2),
    transaction_date Date,
    created_at DateTime,
    status String,
    ingestion_date Date DEFAULT today()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(transaction_date)
ORDER BY (transaction_date, customer_id, transaction_id)
"""
client.execute(ddl_bronze)


In [ ]:
# Ingestion from Staging to ClickHouse
# Reading back from Parquet to simulate the decoupling
df_staging = spark.read.parquet(staging_path)

# Convert to list of tuples for ClickHouse driver
# For massive datasets, we would use clickhouse-client via subprocess or specialized format writer
data_to_insert = df_staging.collect()

insert_query = f"INSERT INTO {CLICKHOUSE_DB}.transactions_local (transaction_id, customer_id, amount, transaction_date, created_at, status) VALUES"
client.execute(insert_query, data_to_insert)


In [ ]:
# Validation
result = client.execute(f"SELECT count(), sum(amount) FROM {CLICKHOUSE_DB}.transactions_local")
print(f"Rows inserted: {result[0][0]}")
print(f"Total Amount: {result[0][1]}")

packages = [
    "com.clickhouse.spark:clickhouse-spark-runtime-3.4_2.12:0.8.0",
    "com.clickhouse:clickhouse-client:0.7.0",
    "com.clickhouse:clickhouse-http-client:0.7.0",
    "com.oracle.database.jdbc:ojdbc8:23.2.0.0",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1"
]

spark = (
    SparkSession.builder
        .appName("OracleToClickHouse")
        .master("spark://spark-master:7077")
        .config("spark.jars.packages", ",".join(packages))
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
)

In [ ]:
oracle_df = (
    spark.read.format("jdbc")
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("url", "jdbc:oracle:thin:@//ORACLE_HOST:1521/SERVICE_NAME")
    .option("user", "ORACLE_USER")
    .option("password", "ORACLE_PASS")
    .option("dbtable", "SCHEMA.TABELA_FONTE")
    .load()
)

oracle_df.show(5)